# Etapa 3: evaluación de la red GCN-DeepGPD

Se prueban los modelos de O3 y SO2 que se entrenaron con la red.
Para cada contaminante se compara la red contra la **GPD constante** (un solo $(\xi, \sigma)$ para todos los excesos de entrenamiento, `src/gpd_bfgs.py`).

Se calculan los criterios de evaluación de la Sección 3.7 del reporte:
1. Log-verosimilitud negativa (NLL) de los excesos: red vs GPD constante.
2. Puntaje de Brier de $P(Y>u+x \mid Y>u)$, comparado con predecir siempre la frecuencia de entrenamiento.
3. Calibración de las probabilidades (5 grupos).
4. Aciertos y falsas alarmas al emitir una alerta.

Se evalúa en los tres periodos: entrenamiento (2023-2024), validación (2025) y prueba (2026).

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import torch
from scipy.stats import genpareto
from sklearn.metrics import roc_auc_score


if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())

from src.train import prepare_hourly_data, STATION_COORDS
from src.gpd_bfgs import fit_gpd_bfgs
from utils.model_architecture import Combiner
from utils.model_params import params_dict
from utils.haversine_distance import haversine_matrix
from utils.train_utils import predict

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", dispositivo)

Dispositivo: cuda



`u` es el umbral del modelo y `x` la distancia a la norma (`u + x`). el csv es el archivo de predicciones (periodo de entrenamiento), se usa para verificar que los modelos del repo dan lo mismo

In [3]:
contaminantes = {
    "O3":  {"u": 63.0, "x": 27.0, "modelo": "models/model_o3.pth",
            "csv": "data/predictions_O3_probs.csv"},
    "SO2": {"u": 20.2, "x": 54.8, "modelo": "models/model_so2_hourly.pth",
            "csv": "data/predictions_SO2_hourly_probs.csv"},
}

D = haversine_matrix(STATION_COORDS)

## Predicciones de la red en los tres periodos

`prepare_hourly_data` regresa (entrenamiento, 2025, 2026). Cada dato trae los excesos observados `y` (en ppb, `NaN` donde no hubo exceso).
La red da un $(\xi, \sigma)$ por estación y día. Como `prepare_hourly_data` vuelve a guardar `models/scaler.pkl`, guardo una copia y
la regreso al final para no cambiar ese archivo.

In [4]:
with open("models/scaler.pkl", "rb") as f:
    scaler_original = f.read()

resultados_red = {}

for pol in contaminantes:
    u = contaminantes[pol]["u"]
    print("=====", pol, "=====")
    train, test, val = prepare_hourly_data("data/datos_imputados.csv", pol, u)

    modelo = Combiner(
        ds_params=params_dict["ds_params"],
        gcn_params=params_dict["gcn_params"],
        graph_params=params_dict["graph_params"],
    )
    modelo.load_state_dict(torch.load(contaminantes[pol]["modelo"], map_location="cpu"))

    periodos = {"entrenamiento (2023-2024)": train,
                "validacion (2025)": test,
                "prueba (2026)": val}

    resultados_red[pol] = {}
    for nombre in periodos:
        datos = periodos[nombre]
        pred = predict(modelo, datos, D, device=dispositivo)     # (dias, 15 estaciones, 2)
        y = datos["y"].cpu().numpy()[:, 0, :, :]                 # (dias, 15, 24)
        resultados_red[pol][nombre] = {"y": y, "xi": pred[:, :, 0], "sigma": pred[:, :, 1]}
        print("  ", nombre, "-> dias:", y.shape[0], "| excesos observados:", int(np.sum(~np.isnan(y))))

with open("models/scaler.pkl", "wb") as f:
    f.write(scaler_original)

===== O3 =====
V torch.Size([730, 17, 15, 24])
S torch.Size([730, 1, 15, 24])
Y torch.Size([730, 1, 15, 24])
V torch.Size([364, 17, 15, 24])
S torch.Size([364, 1, 15, 24])
Y torch.Size([364, 1, 15, 24])
V torch.Size([211, 17, 15, 24])
S torch.Size([211, 1, 15, 24])
Y torch.Size([211, 1, 15, 24])
   entrenamiento (2023-2024) -> dias: 730 | excesos observados: 5153
   validacion (2025) -> dias: 364 | excesos observados: 8596
   prueba (2026) -> dias: 211 | excesos observados: 3919
===== SO2 =====
V torch.Size([730, 17, 15, 24])
S torch.Size([730, 1, 15, 24])
Y torch.Size([730, 1, 15, 24])
V torch.Size([364, 17, 15, 24])
S torch.Size([364, 1, 15, 24])
Y torch.Size([364, 1, 15, 24])
V torch.Size([211, 17, 15, 24])
S torch.Size([211, 1, 15, 24])
Y torch.Size([211, 1, 15, 24])
   entrenamiento (2023-2024) -> dias: 730 | excesos observados: 687
   validacion (2025) -> dias: 364 | excesos observados: 1288
   prueba (2026) -> dias: 211 | excesos observados: 413


Se comparan los $(\xi, \sigma)$ del periodo de entrenamiento con los del archivo de predicciones.

In [5]:
for pol in contaminantes:
    csv = pd.read_csv(contaminantes[pol]["csv"])
    r = resultados_red[pol]["entrenamiento (2023-2024)"]

    dif_xi = np.abs(csv["XI"].values - r["xi"].reshape(-1)).max()
    dif_sigma = np.abs(csv["SIGMA"].values - r["sigma"].reshape(-1)).max()
    print(pol, "- filas del csv:", len(csv), "| diferencia maxima en xi:", round(dif_xi, 6), "| en sigma:", round(dif_sigma, 6))

    if dif_xi < 1e-3 and dif_sigma < 1e-2:
        print("   Los modelos del repositorio reproducen el archivo de predicciones.")
    else:
        print("   OJO: los resultados no coinciden con el archivo de predicciones.")

O3 - filas del csv: 10950 | diferencia maxima en xi: 1e-06 | en sigma: 4.9e-05
   Los modelos del repositorio reproducen el archivo de predicciones.
SO2 - filas del csv: 10950 | diferencia maxima en xi: 1e-06 | en sigma: 0.000219
   Los modelos del repositorio reproducen el archivo de predicciones.


## GPD constante como referencia

Se ajusta una sola GPD a todos los excesos de entrenamiento con `fit_gpd_bfgs`

In [6]:
gpd_constante = {}

for pol in contaminantes:
    y_train = resultados_red[pol]["entrenamiento (2023-2024)"]["y"]
    ajuste = fit_gpd_bfgs(torch.tensor(y_train))
    gpd_constante[pol] = (ajuste["xi"], ajuste["sigma"])
    print(pol, "- excesos de entrenamiento:", ajuste["n_exceed"])
    print("   xi =", round(ajuste["xi"], 3), "(ee", round(ajuste["se_xi"], 3), ") | sigma =", round(ajuste["sigma"], 2), "(ee", round(ajuste["se_sigma"], 2), ")")

O3 - excesos de entrenamiento: 5153
   xi = -0.028 (ee 0.013 ) | sigma = 15.93 (ee 0.3 )
SO2 - excesos de entrenamiento: 687
   xi = 0.251 (ee 0.049 ) | sigma = 11.36 (ee 0.7 )


## Métricas por contaminante y periodo

Cada hora con exceso es una observación. El evento es que el exceso llegue a la norma ($Y - u \geq x$).
La probabilidad de la red es la de su día y estación, la de la GPD constante es la misma para todos.
La "frecuencia" es predecir siempre la proporción de eventos del entrenamiento.

In [7]:
tabla = []
tabla_calibracion = []

for pol in contaminantes:
    x = contaminantes[pol]["x"]
    xi_c, sigma_c = gpd_constante[pol]

    # frecuencia de eventos en entrenamiento
    y_train = resultados_red[pol]["entrenamiento (2023-2024)"]["y"]
    y_train = y_train[~np.isnan(y_train)]
    tasa_train = np.mean(y_train >= x)

    print("==========", pol, "==========")
    print("Frecuencia de eventos en entrenamiento:", round(tasa_train, 4))

    for periodo in resultados_red[pol]:
        y = resultados_red[pol][periodo]["y"]
        hay_exceso = ~np.isnan(y)

        # parametros de la red para cada hora con exceso (se repite el de su dia y estacion)
        xi_todo = np.repeat(resultados_red[pol][periodo]["xi"][:, :, np.newaxis], 24, axis=2)
        sigma_todo = np.repeat(resultados_red[pol][periodo]["sigma"][:, :, np.newaxis], 24, axis=2)
        exc = y[hay_exceso]
        xi = xi_todo[hay_exceso]
        sigma = sigma_todo[hay_exceso]

        evento = (exc >= x).astype(int)
        n_exc = len(exc)
        n_eventos = int(evento.sum())

        # NLL (igual que la perdida del entrenamiento: el argumento del log se recorta en 1e-6)
        arg_red = 1 + xi * exc / sigma
        fuera_red = np.mean(arg_red <= 0)
        arg_red = np.clip(arg_red, 1e-6, None)
        nll_red = np.mean(np.log(sigma) + (1 + 1 / xi) * np.log(arg_red))

        arg_c = np.clip(1 + xi_c * exc / sigma_c, 1e-6, None)
        nll_c = np.mean(np.log(sigma_c) + (1 + 1 / xi_c) * np.log(arg_c))

        # probabilidades de superar la norma, dado que ya se supero u
        p_red = 1 - genpareto.cdf(x, c=xi, scale=sigma)
        p_c = 1 - genpareto.cdf(x, c=xi_c, scale=sigma_c)

        brier_red = np.mean((p_red - evento) ** 2)
        brier_c = np.mean((p_c - evento) ** 2)
        brier_freq = np.mean((tasa_train - evento) ** 2)

        if n_eventos > 0 and n_eventos < n_exc:
            auc = roc_auc_score(evento, p_red)
        else:
            auc = np.nan

        print("")
        print(periodo)
        print("  excesos:", n_exc, "| eventos (>= u + x):", n_eventos, "| tasa:", round(n_eventos / n_exc, 4))
        print("  NLL      red:", round(nll_red, 4), "| GPD constante:", round(nll_c, 4))
        if nll_red < nll_c:
            print("  -> la red tiene menor NLL que la GPD constante")
        else:
            print("  -> la red NO mejora la NLL de la GPD constante")
        print("  Brier    red:", round(brier_red, 4), "| GPD constante:", round(brier_c, 4), "| frecuencia:", round(brier_freq, 4))
        if brier_red < brier_freq:
            print("  -> el Brier de la red es menor que el de la frecuencia (mejora de", round(100 * (1 - brier_red / brier_freq), 1), "%)")
        else:
            print("  -> el Brier de la red NO es menor que el de la frecuencia")
        print("  AUC de la red:", round(auc, 3))
        print("  probabilidad media: red", round(p_red.mean(), 3), "| GPD constante", round(p_c, 3), "| observada", round(n_eventos / n_exc, 3))
        print("  excesos fuera del soporte de la red:", round(100 * fuera_red, 2), "%")

        tabla.append({"contaminante": pol, "periodo": periodo, "excesos": n_exc, "eventos": n_eventos,
                      "tasa_eventos": n_eventos / n_exc, "tasa_entrenamiento": tasa_train,
                      "nll_red": nll_red, "nll_gpd_constante": nll_c,
                      "brier_red": brier_red, "brier_gpd_constante": brier_c, "brier_frecuencia": brier_freq,
                      "auc_red": auc, "prob_media_red": p_red.mean(), "prob_gpd_constante": p_c})

        # calibracion: 5 grupos segun la probabilidad de la red
        datos_cal = pd.DataFrame({"p": p_red, "evento": evento})
        datos_cal["grupo"] = pd.qcut(datos_cal["p"], 5, labels=False, duplicates="drop") + 1
        cal = datos_cal.groupby("grupo").agg(n=("evento", "size"), prob_predicha=("p", "mean"), frec_observada=("evento", "mean"))
        cal = cal.reset_index()
        cal.insert(0, "periodo", periodo)
        cal.insert(0, "contaminante", pol)
        tabla_calibracion.append(cal)

========== O3 ==========
Frecuencia de eventos en entrenamiento: 0.1694

entrenamiento (2023-2024)
  excesos: 5153 | eventos (>= u + x): 873 | tasa: 0.1694
  NLL      red: 3.6974 | GPD constante: 3.7396
  -> la red tiene menor NLL que la GPD constante
  Brier    red: 0.131 | GPD constante: 0.1408 | frecuencia: 0.1407
  -> el Brier de la red es menor que el de la frecuencia (mejora de 6.9 %)
  AUC de la red: 0.709
  probabilidad media: red 0.204 | GPD constante 0.176 | observada 0.169
  excesos fuera del soporte de la red: 0.0 %

validacion (2025)
  excesos: 8596 | eventos (>= u + x): 1742 | tasa: 0.2027
  NLL      red: 3.8089 | GPD constante: 3.8081
  -> la red NO mejora la NLL de la GPD constante
  Brier    red: 0.1596 | GPD constante: 0.1623 | frecuencia: 0.1627
  -> el Brier de la red es menor que el de la frecuencia (mejora de 1.9 %)
  AUC de la red: 0.585
  probabilidad media: red 0.188 | GPD constante 0.176 | observada 0.203
  excesos fuera del soporte de la red: 0.0 %

prueba (2

## Calibración

Si la red dice 20 %, el evento debería ocurrir cerca del 20 % de las veces.

In [8]:
calibracion = pd.concat(tabla_calibracion, ignore_index=True)

for pol in contaminantes:
    for periodo in resultados_red[pol]:
        print(pol, "-", periodo)
        parte = calibracion[(calibracion["contaminante"] == pol) & (calibracion["periodo"] == periodo)]
        print(parte[["grupo", "n", "prob_predicha", "frec_observada"]].round(3).to_string(index=False))
        print("")

O3 - entrenamiento (2023-2024)
 grupo    n  prob_predicha  frec_observada
     1 1031          0.123           0.053
     2 1033          0.168           0.112
     3 1031          0.192           0.126
     4 1031          0.220           0.188
     5 1027          0.319           0.368

O3 - validacion (2025)
 grupo    n  prob_predicha  frec_observada
     1 1724          0.117           0.133
     2 1715          0.161           0.187
     3 1719          0.182           0.205
     4 1722          0.207           0.208
     5 1716          0.271           0.281

O3 - prueba (2026)
 grupo   n  prob_predicha  frec_observada
     1 789          0.139           0.158
     2 779          0.178           0.173
     3 785          0.198           0.201
     4 783          0.222           0.223
     5 783          0.297           0.305

SO2 - entrenamiento (2023-2024)
 grupo   n  prob_predicha  frec_observada
     1 139          0.000           0.000
     2 136          0.002           0.00

## Alertas: aciertos y falsas alarmas

Condicionado a que ya se superó $u$, se emite una alerta cuando la probabilidad de la red es mayor o igual a $\tau$.
Probamos dos valores: la frecuencia de eventos del entrenamiento y el doble de esa frecuencia.

In [9]:
tabla_alertas = []

for pol in contaminantes:
    x = contaminantes[pol]["x"]
    for fila in tabla:
        if fila["contaminante"] == pol:
            tasa_train = fila["tasa_entrenamiento"]

    for periodo in resultados_red[pol]:
        y = resultados_red[pol][periodo]["y"]
        hay_exceso = ~np.isnan(y)
        xi_todo = np.repeat(resultados_red[pol][periodo]["xi"][:, :, np.newaxis], 24, axis=2)
        sigma_todo = np.repeat(resultados_red[pol][periodo]["sigma"][:, :, np.newaxis], 24, axis=2)
        exc = y[hay_exceso]
        p_red = 1 - genpareto.cdf(x, c=xi_todo[hay_exceso], scale=sigma_todo[hay_exceso])
        evento = (exc >= x)

        print(pol, "-", periodo, "(eventos:", int(evento.sum()), ")")
        for tau in [tasa_train, 2 * tasa_train]:
            alerta = p_red >= tau
            aciertos = np.sum(alerta & evento) / np.sum(evento) if np.sum(evento) > 0 else np.nan
            falsas = np.sum(alerta & ~evento) / np.sum(~evento)
            precision = np.sum(alerta & evento) / np.sum(alerta) if np.sum(alerta) > 0 else np.nan
            print("   tau =", round(tau, 3), "| alertas:", int(alerta.sum()), "| aciertos:", round(aciertos, 3),
                  "| falsas alarmas:", round(falsas, 3), "| precision:", round(precision, 3))
            tabla_alertas.append({"contaminante": pol, "periodo": periodo, "tau": tau, "alertas": int(alerta.sum()),
                                  "aciertos": aciertos, "falsas_alarmas": falsas, "precision": precision})

O3 - entrenamiento (2023-2024) (eventos: 873 )
   tau = 0.169 | alertas: 3590 | aciertos: 0.868 | falsas alarmas: 0.662 | precision: 0.211
   tau = 0.339 | alertas: 331 | aciertos: 0.178 | falsas alarmas: 0.041 | precision: 0.468
O3 - validacion (2025) (eventos: 1742 )
   tau = 0.169 | alertas: 5351 | aciertos: 0.708 | falsas alarmas: 0.601 | precision: 0.231
   tau = 0.339 | alertas: 168 | aciertos: 0.036 | falsas alarmas: 0.015 | precision: 0.375
O3 - prueba (2026) (eventos: 832 )
   tau = 0.169 | alertas: 3003 | aciertos: 0.821 | falsas alarmas: 0.752 | precision: 0.227
   tau = 0.339 | alertas: 171 | aciertos: 0.062 | falsas alarmas: 0.039 | precision: 0.304
SO2 - entrenamiento (2023-2024) (eventos: 31 )
   tau = 0.045 | alertas: 195 | aciertos: 0.935 | falsas alarmas: 0.253 | precision: 0.149
   tau = 0.09 | alertas: 125 | aciertos: 0.839 | falsas alarmas: 0.151 | precision: 0.208
SO2 - validacion (2025) (eventos: 69 )
   tau = 0.045 | alertas: 223 | aciertos: 0.304 | falsas alarm

## Resumen y exportación

Se guardan las métricas, la calibración y las alertas en archivos `.csv` dentro de `data/`.

In [10]:
tabla = pd.DataFrame(tabla)
tabla_alertas = pd.DataFrame(tabla_alertas)

print("Resumen de metricas")
print(tabla[["contaminante", "periodo", "excesos", "eventos", "nll_red", "nll_gpd_constante",
             "brier_red", "brier_frecuencia", "auc_red"]].round(3).to_string(index=False))

tabla.to_csv("data/resultados_metricas_modelo.csv", index=False)
calibracion.to_csv("data/resultados_calibracion_modelo.csv", index=False)
tabla_alertas.to_csv("data/resultados_alertas_modelo.csv", index=False)

Resumen de metricas
contaminante                   periodo  excesos  eventos  nll_red  nll_gpd_constante  brier_red  brier_frecuencia  auc_red
          O3 entrenamiento (2023-2024)     5153      873    3.697              3.740      0.131             0.141    0.709
          O3         validacion (2025)     8596     1742    3.809              3.808      0.160             0.163    0.585
          O3             prueba (2026)     3919      832    3.820              3.822      0.165             0.169    0.580
         SO2 entrenamiento (2023-2024)      687       31    3.375              3.681      0.037             0.043    0.912
         SO2         validacion (2025)     1288       69    4.297              3.730      0.067             0.051    0.577
         SO2             prueba (2026)      413       12    3.743              3.514      0.030             0.028    0.495
